## 1. IMPORTS & ENVIRONMENT SETUP

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Deep Learning
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
import timm

# Image Processing
from PIL import Image
import cv2

# Metrics and Utilities
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from sklearn.metrics import top_k_accuracy_score
from sklearn.manifold import TSNE
from tqdm import tqdm
import random

# Set random seeds for reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed(SEED)
torch.backends.cudnn.deterministic = True

# Device configuration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

print("✅ All libraries imported successfully.")

## 2. CONFIGURATION

In [ ]:

!pip install mxnet==1.9.1


In [ ]:

import numpy as np
import os

# --- THE FIX: MONKEY PATCH ---
# We manually restore the 'bool' attribute that newer NumPy versions removed.
# This tricks MXNet into thinking it is running on an old NumPy.
if not hasattr(np, 'bool'):
    np.bool = np.bool_
# -----------------------------

import mxnet as mx
import cv2
from tqdm import tqdm

# --- CONFIGURATION ---
REC_PATH = "/kaggle/input/casia-webface/casia-webface/train.rec"
IDX_PATH = "/kaggle/input/casia-webface/casia-webface/train.idx"
OUTPUT_DIR = "/kaggle/temp/casia_images"

def extract_casia_fixed():
    if not os.path.exists(REC_PATH):
        print(f"❌ Error: Could not find file at {REC_PATH}")
        return

    print(f"🚀 Extracting CASIA-WebFace...")

    try:
        imgrec = mx.recordio.MXIndexedRecordIO(IDX_PATH, REC_PATH, 'r')
        
        # Read Header
        s = imgrec.read_idx(0)
        header, _ = mx.recordio.unpack(s)

        # Get the range of images (Stop before metadata starts)
        max_img_idx = int(header.label[0])

        print(f"📊 Image Indices: 1 → {max_img_idx}")
        print(f"📂 Extracting to: {OUTPUT_DIR}")

        os.makedirs(OUTPUT_DIR, exist_ok=True)

        count = 0
        # Loop strictly over the image range
        for idx in tqdm(range(1, max_img_idx)):
            try:
                s = imgrec.read_idx(idx)
                header, img_bytes = mx.recordio.unpack(s)

                # Decode
                img = mx.image.imdecode(img_bytes).asnumpy()
                img = cv2.cvtColor(img, cv2.COLOR_RGB2BGR)

                # Get Label
                label_id = header.label
                if isinstance(label_id, (list, np.ndarray)):
                    label_id = label_id[0]
                label_str = str(int(label_id))

                # Save
                save_dir = os.path.join(OUTPUT_DIR, label_str)
                os.makedirs(save_dir, exist_ok=True)

                cv2.imwrite(os.path.join(save_dir, f"{idx}.jpg"), img)
                count += 1

            except Exception:
                continue

        print(f"\n✅ Done! Extracted {count} images")
        
    except Exception as e:
        print(f"❌ Critical Error: {e}")

if __name__ == "__main__":
    extract_casia_fixed()

In [ ]:
import random
import matplotlib.pyplot as plt
from PIL import Image
from pathlib import Path

def visualize_clean_samples(clean_path, num_samples=15):
    clean_dir = Path(clean_path)
    # Get all identity subfolders
    all_identities = [d for d in clean_dir.iterdir() if d.is_dir()]
    
    if not all_identities:
        print("❌ No identity folders found in the clean dataset path.")
        return

    # Choose a mix: prioritize your 'new' people if possible
    # (Assuming new people might be at the end or have specific names)
    sample_identities = random.sample(all_identities, min(len(all_identities), num_samples))
    
    fig, axes = plt.subplots(3, 5, figsize=(20, 12))
    axes = axes.ravel()

    print(f"🖼️ Inspecting {len(sample_identities)} samples from cleaned dataset...")

    for i, identity_path in enumerate(sample_identities):
        # Get all images for this person
        imgs = list(identity_path.glob('*.jpg')) + list(identity_path.glob('*.png'))
        
        if imgs:
            sample_img_path = random.choice(imgs)
            img = Image.open(sample_img_path)
            
            axes[i].imshow(img)
            axes[i].set_title(f"{identity_path.name}\nSize: {img.size}", fontsize=10, fontweight='bold')
            axes[i].axis('off')
        else:
            axes[i].text(0.5, 0.5, 'Empty Folder', ha='center')
            axes[i].axis('off')

    plt.suptitle('Verification: Post-Crop & Resized Face Samples (112x112)', fontsize=16, y=1.02)
    plt.tight_layout()
    plt.show()

# Run the visualization
visualize_clean_samples('/kaggle/temp/casia_images')

In [ ]:
class Config:
    # --- HARDWARE ---
    # 2x T4 allows bigger batches. 
    # Bigger batch = More negative samples per step = Better ArcFace learning.
    BATCH_SIZE = 512  # (128 per GPU) - If memory fails, drop to 192.
    NUM_WORKERS = 4   # Max out the CPU loaders to feed the 2 GPUs
    
    # --- DATA ---
    DATA_PATH = '/kaggle/temp/casia_images'
    OUTPUT_PATH = '/kaggle/working/'
    
    # --- ARCHITECTURE ---
    BACKBONE = 'resnet50'
    EMBEDDING_SIZE = 512
    PRETRAINED = True  # Start with ImageNet weights, it saves time
    
    # --- TRAINING DURATION ---
    # CASIA converges fast. 
    # 25 Epochs with Cosine Annealing is the "Sweet Spot" for Kaggle limits.
    NUM_EPOCHS = 50 
    
    # --- OPTIMIZER (SGD IS KING HERE) ---
    # SGD with momentum is standard for state-of-the-art Face Recognition
    OPTIMIZER = 'SGD' 
    LEARNING_RATE = 0.1   # Higher initial LR for SGD (vs 1e-4 for Adam)
    WEIGHT_DECAY = 5e-4   # Standard regularization
    MOMENTUM = 0.9
    
    # --- ARCFACE PARAMETERS (CRITICAL UPDATE) ---
    # For CASIA (10k identities), we need a larger hypersphere (s=64)
    # We DO NOT need progressive growth anymore. 
    # Training is stable enough on CASIA to use fixed parameters.
    ARCFACE_S = 30.0  
    ARCFACE_M = 0.40
    
    # --- SCHEDULER ---
    # "CosineAnnealingLR" is better than ReduceLROnPlateau here.
    # It guarantees we reach a tiny LR by epoch 25, forcing convergence.
    SCHEDULER = 'CosineAnnealing' 
    WARMUP_EPOCHS = 3 # Slowly ramp up LR to avoid exploding gradients at start
    IMG_SIZE = 112
    TRAIN_SPLIT = 0.9
    LABEL_SMOOTHING = 0.1

config = Config()

## 3. DATA EXPLORATION

In [ ]:
# REPLACEMENT FOR SECTION 3 DATA GATHERING
print("=" * 80)
print("DATA GATHERING (FAST VERSION)")
print("=" * 80)

data_path = Path(config.DATA_PATH)
all_data = [] 
image_counts = {}

print(f"📂 Scanning {data_path}...")
# Get class names (folders)
# CASIA folder names are numbers (0, 1, 2...), so we sort them numerically if possible
celebrities = [d.name for d in os.scandir(data_path) if d.is_dir()]
celebrities.sort(key=lambda x: int(x) if x.isdigit() else x) # Numerical sort

num_classes = len(celebrities)
print(f"found {num_classes} identities.")

# Create mapping
name_to_idx = {name: idx for idx, name in enumerate(celebrities)}

# Fast Scraper
from concurrent.futures import ThreadPoolExecutor

def scan_folder(class_name):
    class_dir = os.path.join(config.DATA_PATH, class_name)
    files = []
    # os.scandir is much faster than glob
    with os.scandir(class_dir) as entries:
        for entry in entries:
            if entry.is_file() and entry.name.endswith(('.jpg', '.png', '.jpeg')):
                files.append((entry.path, name_to_idx[class_name]))
    return files

# Use threading to scan 10,000 folders quickly
with ThreadPoolExecutor(max_workers=8) as executor:
    results = list(tqdm(executor.map(scan_folder, celebrities), total=len(celebrities), desc="Indexing Images"))

# Flatten list
for res in results:
    all_data.extend(res)

print(f"✅ Total images indexed: {len(all_data)}")

In [ ]:
# Check the first 5 entries of your class mapping
print(f"First 5 classes: {celebrities[:5]}")

# If they look like ['10', '100', '1000', '11', '12'] (String sort), that is BAD.
# They MUST look like ['0', '1', '2', '3', '4'] (Numerical sort).

## 5. DATA VISUALIZATION - Sample Images

In [ ]:
import random
import matplotlib.pyplot as plt
from PIL import Image
from pathlib import Path

print("🖼️  Displaying sample images (Focusing on NEW data)...")

# 1. Identify New vs Old celebrities
new_data_path = Path(config.DATA_PATH)
old_data_path = Path(config.DATA_PATH)

# Get list of new people
new_celebs = [d.name for d in new_data_path.iterdir() if d.is_dir()]
# Get list of old people (just a sample to fill the rest)
old_celebs = [d.name for d in old_data_path.iterdir() if d.is_dir()]

print(f"   Found {len(new_celebs)} new people to visualize.")

# 2. Build the display list (New people FIRST, then fill with random old ones)
display_count = 15
display_list = new_celebs[:display_count] # Take all new people (up to 15)

# If we have space left, fill with random old celebrities
if len(display_list) < display_count:
    needed = display_count - len(display_list)
    display_list += random.sample(old_celebs, needed)

# 3. Plotting
fig, axes = plt.subplots(3, 5, figsize=(20, 12))
axes = axes.ravel()

for idx, celeb_name in enumerate(display_list):
    # --- FIX: SMART PATH FINDER ---
    # Check if this person is in the NEW folder
    target_path = new_data_path / celeb_name
    
    # If not, check the OLD folder
    if not target_path.exists():
        target_path = old_data_path / celeb_name
    
    # Get images
    images = list(target_path.glob('*.jpg')) + \
             list(target_path.glob('*.png')) + \
             list(target_path.glob('*.jpeg'))
    
    if images:
        sample_img = random.choice(images)
        img = Image.open(sample_img).convert('RGB')
        
        axes[idx].imshow(img)
        
        # Add a star (*) to the title if it's a NEW person
        if celeb_name in new_celebs:
            title_text = f"★ {celeb_name} (NEW)"
            color = 'blue'
        else:
            title_text = celeb_name
            color = 'black'
            
        axes[idx].set_title(title_text, fontsize=10, fontweight='bold', color=color)
        axes[idx].axis('off')

plt.suptitle('Dataset Samples (★ = New Data)', fontsize=16, fontweight='bold', y=0.95)
plt.tight_layout()
plt.savefig(os.path.join(config.OUTPUT_PATH, 'sample_images_mixed.png'), dpi=300, bbox_inches='tight')
plt.show()

print("✅ Visualization complete. Check that your new people are in the first row!")

## 6. DATASET CLASS IMPLEMENTATION

In [ ]:
class FaceDataset(Dataset):
    def __init__(self, data_list, celebrity_names=None, transform=None, is_train=True):
        """
        data_list: List of (image_path, label) tuples from split function
        celebrity_names: List of all celebrity names for label mapping (optional)
        """
        self.transform = transform
        self.is_train = is_train
        self.samples = data_list
        
        # Create label mapping (optional, for debugging/visualization)
        if celebrity_names:
            self.idx_to_label = {idx: name for idx, name in enumerate(celebrity_names)}
            self.label_to_idx = {name: idx for idx, name in enumerate(celebrity_names)}
        
        print(f"Loaded {len(self.samples)} images")
    
    def __len__(self):
        return len(self.samples)
    
    def __getitem__(self, idx):
        img_path, label = self.samples[idx]
        
        # Load image
        image = Image.open(img_path).convert('RGB')
        
        # Apply transforms
        if self.transform:
            image = self.transform(image)
        
        return image, label

print("✅ Dataset class defined.")

## 7. DATA AUGMENTATION & TRANSFORMS

In [ ]:
from torchvision.transforms import RandomErasing

def get_train_transform(reduce_augmentation=False):
    """Get training transforms based on training phase"""
    
    if not reduce_augmentation:
        # Full augmentation (standard)
        return transforms.Compose([
            # 1. The "Shift" Trick: Upscale slightly, then random crop
            transforms.Resize((config.IMG_SIZE + 12, config.IMG_SIZE + 12)),
            transforms.RandomCrop(config.IMG_SIZE),
            
            # 2. Mirroring (Doubles data)
            transforms.RandomHorizontalFlip(p=0.5),
            
            # 3. Small rotations (Head tilt)
            transforms.RandomRotation(10), # Reduced from 12 to 10 for stability
            
            # 4. Lighting Robustness (Webcam simulator)
            transforms.ColorJitter(
                brightness=0.2,
                contrast=0.2,
                saturation=0.15,
                hue=0.05
            ),
            
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5]),
            
            # 5. Occlusion robustness
            RandomErasing(p=0.1, scale=(0.02, 0.1)) 
        ])
    else:
        # Reduced augmentation (Fine-tuning phase)
        return transforms.Compose([
            transforms.Resize((config.IMG_SIZE, config.IMG_SIZE)),
            transforms.RandomHorizontalFlip(p=0.5),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5]),
        ])

# Validation (Must match Inference logic exactly)
val_transform = transforms.Compose([
    transforms.Resize((config.IMG_SIZE, config.IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
])

print("✅ Transforms are ready .")

## 8. DATA SPLITTING

## 9. DATA LOADERS

In [ ]:
# ============================================================================
# DATA SPLITTING (Fixed for CASIA & AttributeError)
# ============================================================================
from sklearn.model_selection import train_test_split

print(f"\n🔄 Starting Data Split on {len(all_data)} merged images...")

# 1. Extract labels for stratification
# We need a list of just labels to tell sklearn how to balance the split
all_labels = [label for _, label in all_data]

# 2. Split: Train vs Temp (Val + Test)
# We use config.TRAIN_SPLIT (e.g., 0.9) to keep most data for training
train_data, temp_data, train_labels, temp_labels = train_test_split(
    all_data, 
    all_labels, 
    train_size=config.TRAIN_SPLIT, # Now defined in Config
    stratify=all_labels, # Ensures every person is in the training set
    random_state=SEED
)

# 3. Split: Val vs Test (The "Speed Clip")
# CASIA's "temp_data" (the 10% left) is still huge (~50,000 images).
# Validating on 50k images is too slow for Kaggle.
# We explicitly take only 4000 images for Val and 4000 for Test.
val_data, test_data = train_test_split(
    temp_data,
    train_size=4000, # Limit validation to exactly 4000 images
    test_size=4000,  # Limit test to exactly 4000 images
    stratify=None,   # Stratify might fail on small subsets of 10k classes, so we turn it off
    random_state=SEED
)

print(f"\n📂 Data Split Complete:")
print(f"  • Training samples: {len(train_data)}")
print(f"  • Validation samples: {len(val_data)} (Capped for speed)")
print(f"  • Test samples: {len(test_data)} (Capped for speed)")

# ============================================================================
# CREATE DATASETS & LOADERS
# ============================================================================

# Get initial training transform
train_transform = get_train_transform(reduce_augmentation=False)

# Create Datasets
# using the renamed class 'FaceDataset' (formerly PinsFaceDataset)
train_dataset = FaceDataset(train_data, celebrities, transform=train_transform, is_train=True)
val_dataset = FaceDataset(val_data, celebrities, transform=val_transform, is_train=False)
test_dataset = FaceDataset(test_data, celebrities, transform=val_transform, is_train=False)

print("✅ Datasets created.")

In [ ]:
# Create DataLoaders
train_loader = DataLoader(
    train_dataset,
    batch_size=config.BATCH_SIZE, # This will pick up the new 512 value
    shuffle=True,
    num_workers=config.NUM_WORKERS, # This will pick up the new 4 value
    pin_memory=True,
    drop_last=True # <--- OPTIONAL: You can add this. It prevents a tiny last batch from messing up BatchNorm statistics.
)

val_loader = DataLoader(
    val_dataset,
    batch_size=config.BATCH_SIZE,
    shuffle=False,
    num_workers=config.NUM_WORKERS,
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=config.BATCH_SIZE,
    shuffle=False,
    num_workers=config.NUM_WORKERS,
    pin_memory=True
)

print(f"\n✅ DataLoaders created successfully.")
print(f"   • Train batches: {len(train_loader)}")

## 10. ARCFACE LOSS IMPLEMENTATION

In [ ]:
class ArcMarginProduct(nn.Module):
    """
    Implementation of ArcFace: Additive Angular Margin Loss
    Paper: https://arxiv.org/abs/1801.07698
    """
    def __init__(self, in_features, out_features, s=30.0, m=0.50, easy_margin=False):
        super(ArcMarginProduct, self).__init__()
        self.in_features = in_features
        self.out_features = out_features
        self.s = s
        self.m = m
        self.easy_margin = easy_margin
        
        self.weight = nn.Parameter(torch.FloatTensor(out_features, in_features))
        nn.init.xavier_uniform_(self.weight)
    
    def forward(self, input, label):
        # Normalize features and weights
        cosine = F.linear(F.normalize(input), F.normalize(self.weight))
        
        # Calculate angles
        sine = torch.sqrt(1.0 - torch.pow(cosine, 2))
        phi = cosine * np.cos(self.m) - sine * np.sin(self.m)
        
        if self.easy_margin:
            phi = torch.where(cosine > 0, phi, cosine)
        else:
            phi = torch.where(cosine > np.cos(np.pi - self.m), phi, cosine - np.sin(np.pi - self.m) * self.m)
        
        # Convert label to one-hot
        one_hot = torch.zeros(cosine.size(), device=device)
        one_hot.scatter_(1, label.view(-1, 1).long(), 1)
        
        # Calculate output
        output = (one_hot * phi) + ((1.0 - one_hot) * cosine)
        output *= self.s
        
        return output

print("✅ ArcFace loss implementation defined.")

## 11. MODEL ARCHITECTURE

In [ ]:
class ArcFaceModel(nn.Module):
    def __init__(self, backbone='resnet50', embedding_size=512, num_classes=10575, pretrained=True):
        super(ArcFaceModel, self).__init__()
        
        # 1. Backbone
        if backbone == 'resnet50':
            self.backbone = models.resnet50(pretrained=pretrained)
            in_features = self.backbone.fc.in_features
            self.backbone.fc = nn.Identity()
        elif backbone == 'resnet101':
            self.backbone = models.resnet101(pretrained=pretrained)
            in_features = self.backbone.fc.in_features
            self.backbone.fc = nn.Identity()
        
        # 2. Embedding Head
        self.embedding = nn.Sequential(
            nn.BatchNorm1d(in_features),
            nn.Dropout(0.4),  # Slightly reduced dropout for big data
            nn.Linear(in_features, embedding_size),
            nn.BatchNorm1d(embedding_size),
        )
        
        # 3. ArcFace Head (FIXED)
        # We now use the fixed s=64, m=0.5 from Config
        self.arcface = ArcMarginProduct(
            in_features=embedding_size,
            out_features=num_classes,
            s=config.ARCFACE_S,  # <--- CHANGED: Uses fixed value
            m=config.ARCFACE_M   # <--- CHANGED: Uses fixed value
        )
    
    def forward(self, x, labels=None):
        features = self.backbone(x)
        embeddings = self.embedding(features)
        
        if labels is not None:
            # We must pass the labels to the ArcFace loss layer
            output = self.arcface(embeddings, labels)
            return output, embeddings
        
        return embeddings

print("✅ Model class updated for Fixed ArcFace Strategy.")

## 12. MODEL INITIALIZATION

In [ ]:
print("=" * 80)
print("MODEL INITIALIZATION")
print("=" * 80)

model = ArcFaceModel(
    backbone=config.BACKBONE,
    embedding_size=config.EMBEDDING_SIZE,
    num_classes=num_classes,
    pretrained=config.PRETRAINED
).to(device)

if torch.cuda.device_count() > 1:
    print(f"🚀 Using {torch.cuda.device_count()} GPUs!")
    # This automatically splits the batch across both GPUs
    model = nn.DataParallel(model)
else:
    print("⚠️ Using single GPU")

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"\n🤖 Model: ArcFace with {config.BACKBONE}")
print(f"  • Total parameters: {total_params:,}")
print(f"  • Trainable parameters: {trainable_params:,}")
print(f"  • Embedding size: {config.EMBEDDING_SIZE}")
print(f"  • Number of classes: {num_classes}")

## 13. TRAINING SETUP

In [ ]:
# Loss with label smoothing
criterion = nn.CrossEntropyLoss(label_smoothing=config.LABEL_SMOOTHING)

# REPLACEMENT FOR SECTION 13

# 1. Optimizer: Switch to SGD
optimizer = torch.optim.SGD(
    model.parameters(),
    lr=config.LEARNING_RATE,
    momentum=config.MOMENTUM,
    weight_decay=config.WEIGHT_DECAY
)

# 2. Scheduler: Cosine Annealing
# This smoothly drops LR from 0.1 to 0.00001 over the 25 epochs
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=config.NUM_EPOCHS,
    eta_min=1e-6
)

print(f"✅ Optimizer: SGD (lr={config.LEARNING_RATE})")
print(f"✅ Scheduler: CosineAnnealingLR (25 Epochs)")
print(f"✅ ArcFace: Fixed s={config.ARCFACE_S}, m={config.ARCFACE_M}")


## 14. TRAINING FUNCTIONS

In [ ]:
def train_epoch(model, loader, criterion, optimizer, device, epoch):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    
    # CHANGE: leave=False makes the bar delete itself after the epoch finishes
    pbar = tqdm(loader, desc=f'Epoch {epoch+1}', leave=False)
    
    for images, labels in pbar:
        images, labels = images.to(device), labels.to(device)
        
        optimizer.zero_grad()
        
        # 1. Forward Pass
        outputs, embeddings = model(images, labels)
        
        # 2. Loss
        loss = criterion(outputs, labels)
        loss.backward()
        
        # Clip gradients
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
        optimizer.step()
        
        # 3. Accuracy Calculation (Clean, no margin penalty)
        with torch.no_grad():
            # Handle DataParallel
            if isinstance(model, nn.DataParallel):
                weight = model.module.arcface.weight
                s = model.module.arcface.s
            else:
                weight = model.arcface.weight
                s = model.arcface.s
                
            clean_logits = F.linear(F.normalize(embeddings), F.normalize(weight))
            clean_logits *= s
            
            _, predicted = clean_logits.max(1)
            correct += predicted.eq(labels).sum().item()
            total += labels.size(0)
            
        running_loss += loss.item()
        
        # Update bar
        pbar.set_postfix({
            'loss': f'{loss.item():.4f}',
            'acc': f'{100.*correct/total:.2f}%',
            'lr': f"{optimizer.param_groups[0]['lr']:.2e}"
        })
    
    epoch_loss = running_loss / len(loader)
    epoch_acc = 100. * correct / total
    
    return epoch_loss, epoch_acc

def validate(model, loader, criterion, device):
    """Validation function robust to DataParallel"""
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0
    
    # --- THE FIX: Handle DataParallel ---
    # If using 2 GPUs, we must access attributes via 'model.module'
    if isinstance(model, nn.DataParallel):
        arcface_layer = model.module.arcface
    else:
        arcface_layer = model.arcface
    
    with torch.no_grad():
        pbar = tqdm(loader, desc='Validation')
        for images, labels in pbar:
            images, labels = images.to(device), labels.to(device)
            
            # 1. Get Embeddings 
            # (DataParallel handles the forward pass automatically)
            embeddings = model(images)
            
            # 2. Calculate Logits manually for accuracy
            # We use the unwrapped 'arcface_layer' we defined above
            cosine = F.linear(F.normalize(embeddings), F.normalize(arcface_layer.weight))
            outputs = cosine * arcface_layer.s
            
            # 3. Loss & Accuracy
            loss = criterion(outputs, labels)
            
            running_loss += loss.item()
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()
            
            pbar.set_postfix({
                'loss': f'{loss.item():.4f}',
                'acc': f'{100.*correct/total:.2f}%'
            })
    
    epoch_loss = running_loss / len(loader)
    epoch_acc = 100. * correct / total
    
    return epoch_loss, epoch_acc

print("✅ Training functions defined ")

## 15. TRAINING LOOP

In [ ]:
print("=" * 80)
print("OPTIMIZED TRAINING - TARGET: 98%+")
print("=" * 80)

history = {
    'train_loss': [], 'train_acc': [],
    'val_loss': [], 'val_acc': [],
    'lr': [] # We don't need to track s/m anymore since they are fixed
}

best_val_acc = 0.0
patience_counter = 0

for epoch in range(config.NUM_EPOCHS):
    print(f"\nEpoch {epoch+1}/{config.NUM_EPOCHS}")
    print("-" * 80)
    
    # 1. Train
    # Note: We removed the "Reduced Augmentation" switch. 
    # We want strong augmentation for the whole training run.
    train_loss, train_acc = train_epoch(
        model, train_loader, criterion, optimizer, device, epoch
    )
    
    # 2. Validate
    val_loss, val_acc = validate(model, val_loader, criterion, device)
    
    # 3. Update Scheduler (Cosine Annealing uses the epoch, not accuracy)
    scheduler.step()
    
    # 4. Save History
    current_lr = optimizer.param_groups[0]['lr']
    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)
    history['lr'].append(current_lr)
    
    print(f"\n📊 Epoch {epoch+1} Summary:")
    print(f"  • Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.2f}%")
    print(f"  • Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.2f}%")
    print(f"  • Learning Rate: {current_lr:.2e}")
    print(f"  • Gap (Train-Val): {train_acc - val_acc:.2f}%")
    
    # 5. Save Best Model
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        patience_counter = 0
        
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'scheduler_state_dict': scheduler.state_dict(),
            'val_acc': val_acc,
            'history': history,
            'config': { # Save config for inference later
                'backbone': config.BACKBONE,
                'embedding_size': config.EMBEDDING_SIZE
            }
        }, os.path.join(config.OUTPUT_PATH, 'best_model.pth'))
        print(f"  ✅ Best model saved! (Val Acc: {val_acc:.2f}%)")
    else:
        # We don't strictly need early stopping for CASIA (25 epochs is safe), 
        # but it's good to keep just in case.
        patience_counter += 1
        
    # Checkpoint every 5 epochs (Good for long runs)
    if (epoch + 1) % 5 == 0:
        checkpoint_path = os.path.join(config.OUTPUT_PATH, f'checkpoint_epoch_{epoch+1}.pth')
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'val_acc': val_acc,
        }, checkpoint_path)
        print(f"  💾 Checkpoint saved at epoch {epoch+1}")

# Training completed
print("\n" + "=" * 80)
print(f"TRAINING COMPLETED! Best Accuracy: {best_val_acc:.2f}%")
print("=" * 80)

In [ ]:
# ============================================================================
# QUICK VISUALIZATION (FIXED)
# ============================================================================
print("\n📊 Generating quick training plot...")

# --- FIX: Recalculate best stats from history so it never crashes ---
best_val_acc = max(history['val_acc'])
best_epoch = history['val_acc'].index(best_val_acc) + 1
# ------------------------------------------------------------------

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
epochs_range = range(1, len(history['train_acc']) + 1)

# Accuracy
axes[0].plot(epochs_range, history['train_acc'], 'b-', label='Train', linewidth=2)
axes[0].plot(epochs_range, history['val_acc'], 'r-', label='Val', linewidth=2)
axes[0].axhline(y=93, color='green', linestyle='--', alpha=0.5, label='93% Goal')
axes[0].scatter(best_epoch, best_val_acc, color='gold', s=200, zorder=5, 
               edgecolor='black', linewidth=2)
axes[0].set_xlabel('Epoch', fontsize=12)
axes[0].set_ylabel('Accuracy (%)', fontsize=12)
axes[0].set_title('Training Progress', fontsize=14, fontweight='bold')
axes[0].legend()
axes[0].grid(alpha=0.3)

# Loss
axes[1].plot(epochs_range, history['train_loss'], 'b-', label='Train', linewidth=2)
axes[1].plot(epochs_range, history['val_loss'], 'r-', label='Val', linewidth=2)
axes[1].set_xlabel('Epoch', fontsize=12)
axes[1].set_ylabel('Loss', fontsize=12)
axes[1].set_title('Loss Progression', fontsize=14, fontweight='bold')
axes[1].legend()
axes[1].grid(alpha=0.3)

# Learning Rate
axes[2].semilogy(epochs_range, history['lr'], 'g-', linewidth=2)
axes[2].set_xlabel('Epoch', fontsize=12)
axes[2].set_ylabel('Learning Rate (log)', fontsize=12)
axes[2].set_title('LR Schedule', fontsize=14, fontweight='bold')
axes[2].grid(alpha=0.3)

plt.suptitle(f'Training Summary - Best: {best_val_acc:.2f}% at Epoch {best_epoch}', 
            fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(config.OUTPUT_PATH, 'training_summary.png'), 
           dpi=300, bbox_inches='tight')
plt.show()
print("✅ Training plot saved to 'training_summary.png'")

# ============================================================================
# RECOMMENDATIONS BASED ON RESULTS
# ============================================================================
print("\n" + "=" * 80)
print("💡 NEXT STEPS RECOMMENDATIONS")
print("=" * 80)

if best_val_acc >= 93:
    print("""
    🎉 EXCELLENT! You've reached the 93% goal!
    Next steps:
    1. Evaluate on test set to verify generalization
    2. Consider test-time augmentation for extra boost
    3. Try ensemble with multiple checkpoints
    4. Ready for production deployment!
    """)
elif best_val_acc >= 91:
    print(f"""
    💪 VERY GOOD! You're at {best_val_acc:.2f}% - Close to the goal!
    Recommended fine-tuning approach:
    1. Load best model and train 10-15 more epochs
    2. Use very low LR (5e-6)
    3. Slightly increase ArcFace to s=30 (if lowered) or m=0.5
    """)
elif best_val_acc >= 80:
    print(f"""
    ✅ GOOD PROGRESS at {best_val_acc:.2f}%!
    To reach 93%:
    1. Fine-tune with stronger ArcFace margin (m=0.5)
    2. Train 15-20 more epochs with LR=1e-5
    3. Ensure augmentation is strong
    """)
else:
    print(f"""
    📊 Current: {best_val_acc:.2f}% - Needs improvement
    Suggestions:
    1. Ensure Class Sorting (0, 1, 2...) is correct!
    2. Check ArcFace parameters (Current: s={config.ARCFACE_S}, m={config.ARCFACE_M})
    3. Increase training epochs (50+ for CASIA)
    """)
print("=" * 80)
print("🏁 TRAINING SESSION COMPLETE")
print("=" * 80)

## 16. TRAINING VISUALIZATION

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.gridspec import GridSpec
import numpy as np
from scipy.ndimage import uniform_filter1d
import os

print("📊 Generating publication-quality charts...")

# --- SAFETY: Recalculate stats to prevent crashes ---
# This ensures variables exist even if you restart the kernel
best_val_acc = max(history['val_acc'])
best_epoch = history['val_acc'].index(best_val_acc) + 1
final_train = history['train_acc'][-1]
final_val = history['val_acc'][-1]
epochs = range(1, len(history['train_acc']) + 1)
# ----------------------------------------------------

# Create figure with custom layout
fig = plt.figure(figsize=(24, 16))
gs = GridSpec(4, 3, figure=fig, hspace=0.3, wspace=0.25)

# Professional Color Scheme (Paper-Friendly)
train_color = '#2980b9' # Strong Blue
val_color = '#c0392b'   # Strong Red
lr_color = '#27ae60'    # Green
gap_color = '#8e44ad'   # Purple

# ============== Main Plots ==============

# 1. Loss Plot (with smoothing)
ax1 = fig.add_subplot(gs[0, 0])
ax1.plot(epochs, history['train_loss'], alpha=0.3, color=train_color, linewidth=1)
ax1.plot(epochs, history['val_loss'], alpha=0.3, color=val_color, linewidth=1)
# Add smoothing for cleaner paper visuals
if len(history['train_loss']) > 5:
    smooth_train = uniform_filter1d(history['train_loss'], size=5)
    smooth_val = uniform_filter1d(history['val_loss'], size=5)
    ax1.plot(epochs, smooth_train, label='Train Loss (Smoothed)', color=train_color, linewidth=2.5)
    ax1.plot(epochs, smooth_val, label='Val Loss (Smoothed)', color=val_color, linewidth=2.5)
ax1.set_xlabel('Epoch', fontsize=12, fontweight='bold')
ax1.set_ylabel('Loss', fontsize=12, fontweight='bold')
ax1.set_title('A. Loss Progression', fontsize=14, fontweight='bold', loc='left')
ax1.legend(loc='upper right', frameon=True)
ax1.grid(alpha=0.2, linestyle='--')

# 2. Accuracy Plot with Annotations
ax2 = fig.add_subplot(gs[0, 1])
ax2.plot(epochs, history['train_acc'], label='Train Acc', color=train_color, linewidth=2)
ax2.plot(epochs, history['val_acc'], label='Val Acc', color=val_color, linewidth=2)

# Highlight Best Epoch
ax2.scatter(best_epoch, best_val_acc, color='gold', s=150, zorder=5, edgecolor='black', linewidth=2)
ax2.annotate(f'Peak: {best_val_acc:.1f}%', 
             xy=(best_epoch, best_val_acc), 
             xytext=(best_epoch, best_val_acc - 15),
             fontsize=11, fontweight='bold',
             bbox=dict(boxstyle="round,pad=0.3", fc="white", ec="black", alpha=0.8),
             arrowprops=dict(arrowstyle='->', color='black', lw=1.5))

ax2.set_xlabel('Epoch', fontsize=12, fontweight='bold')
ax2.set_ylabel('Accuracy (%)', fontsize=12, fontweight='bold')
ax2.set_title('B. Accuracy Trajectory', fontsize=14, fontweight='bold', loc='left')
ax2.legend(loc='lower right')
ax2.grid(alpha=0.2, linestyle='--')


# 3. Learning Rate (Cosine Decay Visualization)
ax3 = fig.add_subplot(gs[0, 2])
ax3.plot(epochs, history['lr'], color=lr_color, linewidth=2.5)
ax3.fill_between(epochs, 0, history['lr'], alpha=0.1, color=lr_color)
ax3.set_xlabel('Epoch', fontsize=12, fontweight='bold')
ax3.set_ylabel('Learning Rate', fontsize=12, fontweight='bold')
ax3.set_title('C. LR Schedule (Cosine Decay)', fontsize=14, fontweight='bold', loc='left')
ax3.grid(alpha=0.2)
# Use sci notation for y-axis
ax3.ticklabel_format(axis='y', style='sci', scilimits=(0,0))


# 4. Overfitting Monitor (Gap Analysis)
ax4 = fig.add_subplot(gs[1, 0])
accuracy_gap = [t - v for t, v in zip(history['train_acc'], history['val_acc'])]
ax4.plot(epochs, accuracy_gap, color=gap_color, linewidth=2)
ax4.fill_between(epochs, 0, accuracy_gap, where=[g >= 0 for g in accuracy_gap], 
                 color='red', alpha=0.1, label='Overfitting (Train > Val)')
ax4.fill_between(epochs, 0, accuracy_gap, where=[g < 0 for g in accuracy_gap], 
                 color='green', alpha=0.1, label='Robustness (Val > Train)')
ax4.axhline(y=0, color='black', linestyle='-', alpha=0.3)
ax4.set_xlabel('Epoch', fontsize=12, fontweight='bold')
ax4.set_ylabel('Gap (%)', fontsize=12, fontweight='bold')
ax4.set_title('D. Generalization Gap', fontsize=14, fontweight='bold', loc='left')
ax4.legend(loc='upper left')
ax4.grid(alpha=0.2)


# 5. Improvement Rate (Derivative of Validation Accuracy)
ax5 = fig.add_subplot(gs[1, 1])
val_improvement = np.diff(history['val_acc'], prepend=history['val_acc'][0])
# Color bars: Green = Positive gain, Red = Regression
colors = ['#27ae60' if imp > 0 else '#c0392b' for imp in val_improvement]
ax5.bar(epochs, val_improvement, color=colors, alpha=0.6, width=0.8)
ax5.axhline(y=0, color='black', linewidth=1)
ax5.set_xlabel('Epoch', fontsize=12, fontweight='bold')
ax5.set_ylabel('Accuracy Gain (%)', fontsize=12, fontweight='bold')
ax5.set_title('E. Epoch-wise Improvement', fontsize=14, fontweight='bold', loc='left')
ax5.grid(alpha=0.2, axis='y')


# 6. Validation Accuracy Histogram
ax6 = fig.add_subplot(gs[1, 2])
ax6.hist(history['val_acc'], bins=15, color=val_color, alpha=0.7, edgecolor='black', rwidth=0.9)
ax6.axvline(x=np.mean(history['val_acc']), color='blue', linestyle='--', linewidth=2, 
            label=f'Mean: {np.mean(history["val_acc"]):.1f}%')
ax6.axvline(x=best_val_acc, color='gold', linestyle='--', linewidth=2,
            label=f'Max: {best_val_acc:.1f}%')
ax6.set_xlabel('Accuracy (%)', fontsize=12, fontweight='bold')
ax6.set_ylabel('Frequency', fontsize=12, fontweight='bold')
ax6.set_title('F. Performance Distribution', fontsize=14, fontweight='bold', loc='left')
ax6.legend()
ax6.grid(alpha=0.2, axis='y')


# 7. Training Strategy Visualization (UPDATED for Cosine Annealing)
ax7 = fig.add_subplot(gs[2, :])
ax7.plot(epochs, history['train_acc'], label='Training', color=train_color, linewidth=2.5, alpha=0.8)
ax7.plot(epochs, history['val_acc'], label='Validation', color=val_color, linewidth=2.5)

# Dynamic Phases based on Total Epochs
total_epochs = len(epochs)
p1 = int(total_epochs * 0.3)
p2 = int(total_epochs * 0.7)

# Phase 1: High Learning
ax7.axvspan(0, p1, alpha=0.05, color='green')
ax7.text(p1/2, 5, "Phase I: Rapid Learning\n(High LR)", ha='center', fontsize=10, fontweight='bold')

# Phase 2: Consolidation
ax7.axvspan(p1, p2, alpha=0.05, color='orange')
ax7.text((p1+p2)/2, 5, "Phase II: Feature Consolidation\n(Medium LR)", ha='center', fontsize=10, fontweight='bold')

# Phase 3: Fine-tuning
ax7.axvspan(p2, total_epochs, alpha=0.05, color='purple')
ax7.text((p2+total_epochs)/2, 5, "Phase III: Convergence\n(Low LR)", ha='center', fontsize=10, fontweight='bold')

ax7.set_xlabel('Epoch', fontsize=12, fontweight='bold')
ax7.set_ylabel('Accuracy (%)', fontsize=12, fontweight='bold')
ax7.set_title('G. Complete Training Lifecycle', fontsize=14, fontweight='bold', loc='left')
ax7.legend(loc='lower right', frameon=True, fontsize=11)
ax7.grid(alpha=0.2)
ax7.set_xlim(1, total_epochs)


# 8. Statistical Summary Card
ax8 = fig.add_subplot(gs[3, :])
ax8.axis('off')

stats_text = f"""
🔬 EXPERIMENTAL SUMMARY REPORT
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
MODEL CONFIGURATION:
• Backbone:       {config.BACKBONE} (Pretrained)
• Head:           ArcFace (s={config.ARCFACE_S}, m={config.ARCFACE_M})
• Optimizer:      SGD + Cosine Annealing
• Batch Size:     {config.BATCH_SIZE}

PERFORMANCE METRICS:
• Peak Validation Accuracy:   {best_val_acc:.2f}% (Epoch {best_epoch})
• Final Training Accuracy:    {final_train:.2f}%
• Final Generalization Gap:   {final_train - final_val:.2f}%
• Convergence Stability:      {np.std(history['val_acc'][-5:]):.2f} (Std Dev last 5 epochs)

ANALYSIS:
• The model achieved a peak accuracy of {best_val_acc:.2f}%.
• Total improvement from initialization: +{best_val_acc - history['val_acc'][0]:.2f}%.
"""

ax8.text(0.02, 0.5, stats_text, transform=ax8.transAxes, 
         fontsize=12, fontfamily='monospace', verticalalignment='center')

# Save
plt.tight_layout()
plt.savefig(os.path.join(config.OUTPUT_PATH, 'paper_ready_visualizations.png'), 
            dpi=300, bbox_inches='tight', facecolor='white')
plt.show()

print("✅ Paper-ready visualizations saved to 'paper_ready_visualizations.png'")

In [ ]:
print("📈 Generating training visualizations...")

fig, axes = plt.subplots(1, 3, figsize=(20, 5))

# Loss plot
axes[0].plot(history['train_loss'], label='Train Loss', linewidth=2, marker='o', markersize=4)
axes[0].plot(history['val_loss'], label='Val Loss', linewidth=2, marker='s', markersize=4)
axes[0].set_xlabel('Epoch', fontsize=12)
axes[0].set_ylabel('Loss', fontsize=12)
axes[0].set_title('Training and Validation Loss', fontsize=14, fontweight='bold')
axes[0].legend()
axes[0].grid(alpha=0.3)

# Accuracy plot
axes[1].plot(history['train_acc'], label='Train Acc', linewidth=2, marker='o', markersize=4)
axes[1].plot(history['val_acc'], label='Val Acc', linewidth=2, marker='s', markersize=4)
axes[1].set_xlabel('Epoch', fontsize=12)
axes[1].set_ylabel('Accuracy (%)', fontsize=12)
axes[1].set_title('Training and Validation Accuracy', fontsize=14, fontweight='bold')
axes[1].legend()
axes[1].grid(alpha=0.3)

# Learning rate plot
axes[2].plot(history['lr'], linewidth=2, marker='o', markersize=4, color='green')
axes[2].set_xlabel('Epoch', fontsize=12)
axes[2].set_ylabel('Learning Rate', fontsize=12)
axes[2].set_title('Learning Rate Schedule', fontsize=14, fontweight='bold')
axes[2].set_yscale('log')
axes[2].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(config.OUTPUT_PATH, 'training_history.png'), dpi=300, bbox_inches='tight')
plt.show()

print("✅ Training history saved.")

In [ ]:
def plot_complete_training_history(history):
    """Plot the complete training history including all sessions"""
    fig, axes = plt.subplots(2, 2, figsize=(15, 10))
    
    epochs = range(1, len(history['train_acc']) + 1)
    
    # Accuracy plot
    axes[0, 0].plot(epochs, history['train_acc'], label='Train', color='blue', alpha=0.7)
    axes[0, 0].plot(epochs, history['val_acc'], label='Validation', color='orange', linewidth=2)
    axes[0, 0].set_title('Accuracy Over All Training Sessions')
    axes[0, 0].set_xlabel('Total Epochs')
    axes[0, 0].set_ylabel('Accuracy (%)')
    axes[0, 0].legend()
    axes[0, 0].grid(True, alpha=0.3)
    axes[0, 0].axvline(x=35, color='red', linestyle='--', alpha=0.5, label='Fine-tuning Start')
    
    # Loss plot
    axes[0, 1].plot(epochs, history['train_loss'], label='Train', color='blue', alpha=0.7)
    axes[0, 1].plot(epochs, history['val_loss'], label='Validation', color='orange', linewidth=2)
    axes[0, 1].set_title('Loss Over All Training Sessions')
    axes[0, 1].set_xlabel('Total Epochs')
    axes[0, 1].set_ylabel('Loss')
    axes[0, 1].legend()
    axes[0, 1].grid(True, alpha=0.3)
    
    # Learning rate plot
    axes[1, 0].plot(epochs, history['lr'], color='green', linewidth=2)
    axes[1, 0].set_title('Learning Rate Schedule')
    axes[1, 0].set_xlabel('Total Epochs')
    axes[1, 0].set_ylabel('Learning Rate')
    axes[1, 0].set_yscale('log')
    axes[1, 0].grid(True, alpha=0.3)
    
    # Accuracy difference (train - val)
    acc_diff = [t - v for t, v in zip(history['train_acc'], history['val_acc'])]
    axes[1, 1].plot(epochs, acc_diff, color='purple', linewidth=2)
    axes[1, 1].axhline(y=0, color='black', linestyle='-', alpha=0.3)
    axes[1, 1].fill_between(epochs, 0, acc_diff, alpha=0.3, color='purple')
    axes[1, 1].set_title('Overfitting Monitor (Train - Val Accuracy)')
    axes[1, 1].set_xlabel('Total Epochs')
    axes[1, 1].set_ylabel('Accuracy Difference (%)')
    axes[1, 1].grid(True, alpha=0.3)
    
    plt.suptitle('Complete Training History: Initial + Fine-tuning', fontsize=16)
    plt.tight_layout()
    plt.show()

# Use after training
plot_complete_training_history(history)

## 17. MODEL EVALUATION FUNCTIONS

In [ ]:
print("=" * 80)
print("MODEL EVALUATION")
print("=" * 80)

# Load best model
checkpoint = torch.load(os.path.join(config.OUTPUT_PATH, 'best_model.pth'))
model.load_state_dict(checkpoint['model_state_dict'])
print(f"\n✅ Loaded best model from epoch {checkpoint['epoch']+1}")

model.eval()

def evaluate_model(model, loader, device):
    all_preds = []
    all_labels = []
    all_embeddings = []
    
    with torch.no_grad():
        for images, labels in tqdm(loader, desc='Evaluating'):
            images = images.to(device)
            
            embeddings = model(images)
            
            # Compute logits
            cosine = F.linear(F.normalize(embeddings), F.normalize(model.arcface.weight))
            outputs = cosine * config.ARCFACE_S
            
            _, predicted = outputs.max(1)
            
            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels.numpy())
            all_embeddings.append(embeddings.cpu().numpy())
    
    all_embeddings = np.vstack(all_embeddings)
    
    return np.array(all_preds), np.array(all_labels), all_embeddings

print("✅ Evaluation functions defined.")

## 18. EVALUATE ON TEST SET

In [ ]:
# Evaluate on test set
test_preds, test_labels, test_embeddings = evaluate_model(model, test_loader, device)

# Calculate metrics
test_acc = accuracy_score(test_labels, test_preds) * 100
print(f"\n🎯 Test Accuracy: {test_acc:.2f}%")

# Classification report
print("\n📋 Classification Report:")
class_names = [test_dataset.idx_to_label[i] for i in sorted(test_dataset.idx_to_label.keys())]
print(classification_report(test_labels, test_preds, target_names=class_names, zero_division=0))

## 19. CONFUSION MATRIX VISUALIZATION

In [ ]:
print("\n📊 Generating confusion matrix...")

cm = confusion_matrix(test_labels, test_preds)

plt.figure(figsize=(20, 18))
sns.heatmap(cm, annot=False, fmt='d', cmap='Blues', cbar=True,
            xticklabels=[test_dataset.idx_to_label[i] for i in sorted(test_dataset.idx_to_label.keys())],
            yticklabels=[test_dataset.idx_to_label[i] for i in sorted(test_dataset.idx_to_label.keys())])
plt.xlabel('Predicted', fontsize=14)
plt.ylabel('True', fontsize=14)
plt.title('Confusion Matrix - Test Set', fontsize=16, fontweight='bold')
plt.xticks(rotation=90, fontsize=8)
plt.yticks(rotation=0, fontsize=8)
plt.tight_layout()
plt.savefig(os.path.join(config.OUTPUT_PATH, 'confusion_matrix.png'), dpi=300, bbox_inches='tight')
plt.show()

# 16. EMBEDDING VISUALIZATION (t-SNE)

In [ ]:

print("\n🎨 Generating t-SNE visualization...")

from sklearn.manifold import TSNE

# Sample embeddings for visualization (to avoid memory issues)
sample_size = min(5000, len(test_embeddings))
indices = np.random.choice(len(test_embeddings), sample_size, replace=False)
sample_embeddings = test_embeddings[indices]
sample_labels = test_labels[indices]

# Apply t-SNE
tsne = TSNE(n_components=2, random_state=SEED, perplexity=30, n_iter=1000)
embeddings_2d = tsne.fit_transform(sample_embeddings)

# Plot
plt.figure(figsize=(16, 12))
scatter = plt.scatter(embeddings_2d[:, 0], embeddings_2d[:, 1], 
                     c=sample_labels, cmap='tab20', alpha=0.6, s=20)
plt.colorbar(scatter, label='Celebrity ID')
plt.xlabel('t-SNE Component 1', fontsize=12)
plt.ylabel('t-SNE Component 2', fontsize=12)
plt.title('t-SNE Visualization of Face Embeddings', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(config.OUTPUT_PATH, 'tsne_embeddings.png'), dpi=300, bbox_inches='tight')
plt.show()

# 17. PER-CLASS ACCURACY

In [ ]:

print("\n📊 Calculating per-class accuracy...")

per_class_correct = {}
per_class_total = {}

for pred, label in zip(test_preds, test_labels):
    if label not in per_class_total:
        per_class_total[label] = 0
        per_class_correct[label] = 0
    
    per_class_total[label] += 1
    if pred == label:
        per_class_correct[label] += 1

# Calculate per-class accuracy
per_class_acc = {}
for label in per_class_total:
    per_class_acc[label] = (per_class_correct[label] / per_class_total[label]) * 100

# Create DataFrame
acc_df = pd.DataFrame([
    {
        'Celebrity': test_dataset.idx_to_label[label],
        'Accuracy (%)': acc,
        'Correct': per_class_correct[label],
        'Total': per_class_total[label]
    }
    for label, acc in per_class_acc.items()
]).sort_values('Accuracy (%)', ascending=False)

print("\n📊 Top 10 Best Performing Classes:")
print(acc_df.head(10).to_string(index=False))

print("\n📊 Bottom 10 Performing Classes:")
print(acc_df.tail(10).to_string(index=False))

# Visualize per-class accuracy
fig, axes = plt.subplots(1, 2, figsize=(20, 8))

# Top 20 classes
top_20_acc = acc_df.head(20)
axes[0].barh(range(len(top_20_acc)), top_20_acc['Accuracy (%)'], color='lightgreen')
axes[0].set_yticks(range(len(top_20_acc)))
axes[0].set_yticklabels(top_20_acc['Celebrity'], fontsize=9)
axes[0].set_xlabel('Accuracy (%)', fontsize=12)
axes[0].set_title('Top 20 Classes by Accuracy', fontsize=14, fontweight='bold')
axes[0].invert_yaxis()
axes[0].grid(axis='x', alpha=0.3)
axes[0].axvline(test_acc, color='red', linestyle='--', linewidth=2, label=f'Overall: {test_acc:.2f}%')
axes[0].legend()

# Bottom 20 classes
bottom_20_acc = acc_df.tail(20)
axes[1].barh(range(len(bottom_20_acc)), bottom_20_acc['Accuracy (%)'], color='lightcoral')
axes[1].set_yticks(range(len(bottom_20_acc)))
axes[1].set_yticklabels(bottom_20_acc['Celebrity'], fontsize=9)
axes[1].set_xlabel('Accuracy (%)', fontsize=12)
axes[1].set_title('Bottom 20 Classes by Accuracy', fontsize=14, fontweight='bold')
axes[1].invert_yaxis()
axes[1].grid(axis='x', alpha=0.3)
axes[1].axvline(test_acc, color='red', linestyle='--', linewidth=2, label=f'Overall: {test_acc:.2f}%')
axes[1].legend()

plt.tight_layout()
plt.savefig(os.path.join(config.OUTPUT_PATH, 'per_class_accuracy.png'), dpi=300, bbox_inches='tight')
plt.show()

# ============================================================================
# 18. ERROR ANALYSIS
# ============================================================================

print("\n" + "=" * 80)
print("ERROR ANALYSIS")
print("=" * 80)

# Find misclassified examples
misclassified_indices = np.where(test_preds != test_labels)[0]
print(f"\n❌ Total misclassified samples: {len(misclassified_indices)}")

# Sample some misclassified examples
if len(misclassified_indices) > 0:
    num_samples = min(15, len(misclassified_indices))
    sample_indices = np.random.choice(misclassified_indices, num_samples, replace=False)
    
    fig, axes = plt.subplots(3, 5, figsize=(20, 12))
    axes = axes.ravel()
    
    for idx, sample_idx in enumerate(sample_indices):
        # Get the image
        img_path, true_label = test_dataset.samples[sample_idx]
        img = Image.open(img_path).convert('RGB')
        
        pred_label = test_preds[sample_idx]
        true_celebrity = test_dataset.idx_to_label[true_label]
        pred_celebrity = test_dataset.idx_to_label[pred_label]
        
        axes[idx].imshow(img)
        axes[idx].set_title(f'True: {true_celebrity}\nPred: {pred_celebrity}', 
                           fontsize=9, color='red')
        axes[idx].axis('off')
    
    plt.suptitle('Misclassified Examples', fontsize=16, fontweight='bold', y=0.98)
    plt.tight_layout()
    plt.savefig(os.path.join(config.OUTPUT_PATH, 'misclassified_examples.png'), dpi=300, bbox_inches='tight')
    plt.show()

# Most confused pairs
confusion_pairs = []
for i in range(len(cm)):
    for j in range(len(cm)):
        if i != j and cm[i][j] > 0:
            confusion_pairs.append({
                'True': test_dataset.idx_to_label[i],
                'Predicted': test_dataset.idx_to_label[j],
                'Count': cm[i][j]
            })

confusion_df = pd.DataFrame(confusion_pairs).sort_values('Count', ascending=False)
print("\n🔀 Top 10 Most Confused Class Pairs:")
print(confusion_df.head(10).to_string(index=False))

# 19. INFERENCE PIPELINE

In [ ]:

print("\n" + "=" * 80)
print("INFERENCE PIPELINE")
print("=" * 80)

def predict_celebrity(model, image_path, transform, device, top_k=5):
    """
    Predict celebrity from an image
    """
    model.eval()
    
    # Load and preprocess image
    image = Image.open(image_path).convert('RGB')
    image_tensor = transform(image).unsqueeze(0).to(device)
    
    with torch.no_grad():
        # Get embedding
        embedding = model(image_tensor)
        
        # Calculate similarities
        cosine = F.linear(F.normalize(embedding), F.normalize(model.arcface.weight))
        logits = cosine * config.ARCFACE_S
        
        # Get top-k predictions
        probs = F.softmax(logits, dim=1)
        top_probs, top_indices = torch.topk(probs, top_k, dim=1)
        
        top_probs = top_probs.cpu().numpy()[0]
        top_indices = top_indices.cpu().numpy()[0]
    
    return top_probs, top_indices, image

# Demonstrate inference on random test images
print("\n🔍 Running inference on sample images...")

num_inference_samples = 6
sample_test_indices = np.random.choice(len(test_dataset), num_inference_samples, replace=False)

fig, axes = plt.subplots(2, 3, figsize=(18, 12))
axes = axes.ravel()

for idx, test_idx in enumerate(sample_test_indices):
    img_path, true_label = test_dataset.samples[test_idx]
    
    # Predict
    top_probs, top_indices, img = predict_celebrity(model, img_path, val_transform, device, top_k=3)
    
    # Get celebrity names
    true_celebrity = test_dataset.idx_to_label[true_label]
    pred_celebrities = [test_dataset.idx_to_label[i] for i in top_indices]
    pred_probs = [f"{p*100:.1f}%" for p in top_probs]
    
    # Display
    axes[idx].imshow(img)
    prediction_text = f"True: {true_celebrity}\n\n"
    prediction_text += f"Top 3 Predictions:\n"
    for i, (celeb, prob) in enumerate(zip(pred_celebrities, pred_probs), 1):
        prediction_text += f"{i}. {celeb}: {prob}\n"
    
    axes[idx].set_title(prediction_text, fontsize=9, ha='left')
    axes[idx].axis('off')

plt.suptitle('Inference Examples with Top-3 Predictions', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(config.OUTPUT_PATH, 'inference_examples.png'), dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
import random
import torch.nn.functional as F
from PIL import Image
import matplotlib.patches as patches

def test_random_predictions(model, test_loader, num_samples=12):
    """Show random predictions with confidence scores"""
    model.eval()
    
    # Get all test data
    all_images = []
    all_labels = []
    all_preds = []
    all_probs = []
    
    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            embeddings = model(images)
            
            # Calculate logits for classification
            cosine = F.linear(F.normalize(embeddings), F.normalize(model.arcface.weight))
            outputs = cosine * model.arcface.s
            
            probs = F.softmax(outputs, dim=1)
            preds = outputs.argmax(dim=1)
            
            all_images.extend(images.cpu())
            all_labels.extend(labels.cpu().numpy())
            all_preds.extend(preds.cpu().numpy())
            all_probs.extend(probs.cpu().numpy())
    
    # Random sample indices
    indices = random.sample(range(len(all_labels)), min(num_samples, len(all_labels)))
    
    # Create visualization
    fig = plt.figure(figsize=(20, 12))
    fig.suptitle('Random Test Predictions', fontsize=16, fontweight='bold')
    
    rows = 3
    cols = 4
    
    for idx, i in enumerate(indices):
        ax = plt.subplot(rows, cols, idx + 1)
        
        # Denormalize image for display
        img = all_images[i].numpy().transpose(1, 2, 0)
        img = img * np.array([0.229, 0.224, 0.225]) + np.array([0.485, 0.456, 0.406])
        img = np.clip(img, 0, 1)
        
        ax.imshow(img)
        ax.axis('off')
        
        true_label = all_labels[i]
        pred_label = all_preds[i]
        confidence = all_probs[i][pred_label] * 100
        
        # Get celebrity names if available
        true_name = celebrities[true_label] if 'celebrities' in globals() else f"Class {true_label}"
        pred_name = celebrities[pred_label] if 'celebrities' in globals() else f"Class {pred_label}"
        
        # Color code: green if correct, red if wrong
        color = 'green' if true_label == pred_label else 'red'
        
        # Create border
        rect = patches.Rectangle((0, 0), img.shape[1]-1, img.shape[0]-1, 
                                linewidth=5, edgecolor=color, facecolor='none')
        ax.add_patch(rect)
        
        # Add text
        title = f"TRUE: {true_name}\nPRED: {pred_name}\nConf: {confidence:.1f}%"
        ax.set_title(title, fontsize=10, color=color, fontweight='bold')
    
    plt.tight_layout()
    plt.show()
    
    # Calculate accuracy on shown samples
    shown_correct = sum(1 for i in indices if all_labels[i] == all_preds[i])
    print(f"\n✅ Accuracy on shown samples: {shown_correct}/{len(indices)} ({shown_correct/len(indices)*100:.1f}%)")

# Run the test
test_random_predictions(model, test_loader, num_samples=12)

In [ ]:
def analyze_prediction_confidence(model, test_loader):
    """Analyze model confidence on correct vs incorrect predictions"""
    model.eval()
    
    correct_confidences = []
    incorrect_confidences = []
    
    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            embeddings = model(images)
            
            cosine = F.linear(F.normalize(embeddings), F.normalize(model.arcface.weight))
            outputs = cosine * model.arcface.s
            probs = F.softmax(outputs, dim=1)
            
            confidences, preds = probs.max(dim=1)
            
            correct_mask = preds == labels
            correct_confidences.extend(confidences[correct_mask].cpu().numpy())
            incorrect_confidences.extend(confidences[~correct_mask].cpu().numpy())
    
    # Create visualization
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    
    # Histogram of confidences
    axes[0].hist(correct_confidences, bins=30, alpha=0.7, label='Correct', color='green', density=True)
    axes[0].hist(incorrect_confidences, bins=30, alpha=0.7, label='Incorrect', color='red', density=True)
    axes[0].set_xlabel('Confidence')
    axes[0].set_ylabel('Density')
    axes[0].set_title('Confidence Distribution')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)
    
    # Box plot
    axes[1].boxplot([correct_confidences, incorrect_confidences], 
                    labels=['Correct', 'Incorrect'],
                    patch_artist=True,
                    boxprops=dict(facecolor='lightblue'))
    axes[1].set_ylabel('Confidence')
    axes[1].set_title('Confidence Comparison')
    axes[1].grid(True, alpha=0.3)
    
    # Statistics
    axes[2].axis('off')
    stats_text = f"""
    📊 CONFIDENCE STATISTICS
    
    ✅ Correct Predictions:
    • Mean Confidence: {np.mean(correct_confidences):.3f}
    • Median: {np.median(correct_confidences):.3f}
    • Std Dev: {np.std(correct_confidences):.3f}
    • Count: {len(correct_confidences)}
    
    ❌ Incorrect Predictions:
    • Mean Confidence: {np.mean(incorrect_confidences):.3f}
    • Median: {np.median(incorrect_confidences):.3f}
    • Std Dev: {np.std(incorrect_confidences):.3f}
    • Count: {len(incorrect_confidences)}
    
    📈 Overall Test Accuracy: {len(correct_confidences)/(len(correct_confidences)+len(incorrect_confidences))*100:.2f}%
    """
    axes[2].text(0.1, 0.5, stats_text, fontsize=11, fontfamily='monospace',
                verticalalignment='center')
    
    plt.suptitle('Model Confidence Analysis', fontsize=16, fontweight='bold')
    plt.tight_layout()
    plt.show()

# Run confidence analysis
analyze_prediction_confidence(model, test_loader)